In [10]:
import pandas as pd
import yaml
from utils.process_data import make_dataset

data = pd.read_csv("input/processed/data.csv", index_col=0, parse_dates=True)

with open("config.yaml") as f:
    config = yaml.safe_load(f)

model_config = config["TreeModel"]
X_cols = model_config["features"] + [f"lag_{i}" for i in model_config["lags"]]
X, y = make_dataset(data, model_config)
X = pd.DataFrame(X, columns=X_cols)
y = pd.Series(y, name=model_config["target_col"])

In [87]:
clusters

array([1, 2, 3, 1], dtype=int32)

In [100]:
distance

array([[0.        , 0.98932311, 0.9766548 , 0.        ],
       [0.98932311, 0.        , 0.9533563 , 0.98932311],
       [0.9766548 , 0.9533563 , 0.        , 0.9766548 ],
       [0.        , 0.98932311, 0.9766548 , 0.        ]])

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

best_err = 1
corr_threshold = 0.8
features = np.array(list("abcd"))
mean_feat_importances = np.array([1000,1,10,100])
X = np.random.randn(100,4)
X[:,0] = X[:,1] * 2

sorted_features = np.argsort(mean_feat_importances)
best_features = features[sorted_features]
print("Sorted features: ", best_features)
corr = np.corrcoef(X, rowvar=False)

distance = 1 - np.abs(corr)
np.fill_diagonal(distance, 0)
linkage_matrix = linkage(squareform(distance), method='average')
clusters = fcluster(linkage_matrix, t=1-corr_threshold, criterion='distance')
keep_features = np.ones_like(sorted_features, dtype=bool)

for i, feat in enumerate(sorted_features):
    # the actual position of the features
    feat_cluster = sorted_features[clusters == clusters[i]]
    other_features = sorted_features[(clusters != clusters[i]) & (keep_features)]
    print("Testing", features[feat_cluster])
    
    if len(feat_cluster) > 1:
        # keep only the feature of the cluster with the highest feat importance 
        test_feat = np.append(other_features, feat_cluster[-1]) 
        drop = sorted_features[feat_cluster[:-1]]
    else:
        test_feat = other_features
        drop = sorted_features[i]
    print("dropping", features[feat_cluster[:-1]])
    break
    
    
    # y_true, y_pred, _ = self.model.rolling_forecast(X[test_feat], y)
    err = 0.8
    
    if err < best_err:
        best_err = err
        keep_features[drop] = False
        best_features = features[sorted_features[keep_features]]
        print(f"New best error {best_err:.4f} with features: {best_features}")

Sorted features:  ['b' 'c' 'd' 'a']
Testing ['b' 'a']
dropping ['b']


In [135]:
drop

array([2])

In [120]:
feat_cluster

array([1, 0])

In [24]:
from utils.process_data import make_dataset
from utils.losses import pinball_loss, quantile_coverage
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from typing import Tuple

class FeatureSelector:

    def __init__(self, model, config):
        
        self.model = model(config, quantiles=[0.5])
        self.features = np.array(config["features"])
    
    
    def recursive_feature_elimination(self, X, y, corr_threshold:float, verbose:bool=True) -> Tuple[np.ndarray, float]:
        y0_true, y0_pred, feat_importances = self.model.rolling_forecast(X, y)
        best_err = pinball_loss(y0_true, y0_pred, 0.5)
        mean_feat_importances = np.mean(feat_importances, axis=0)
        sorted_features = np.argsort(mean_feat_importances)
        best_features = self.features[sorted_features]
        
        if verbose:
            print("Sorted features: ", best_features)

        corr = np.corrcoef(X[sorted_features])
        distance = 1 - np.abs(corr)
        linkage_matrix = linkage(squareform(distance), method='average')
        clusters = fcluster(linkage_matrix, t=1-corr_threshold, criterion='distance')

        keep_features = np.ones_like(sorted_features, dtype=bool)
        for i in range(len(sorted_features)):
            feat_cluster = sorted_features[clusters == clusters[i]]
            other_features = sorted_features[(clusters != clusters[i]) & (keep_features)]
            
            if len(feat_cluster) > 1:
                # keep only the feature of the cluster with the highest feat importance 
                test_feat = np.concat([other_features, feat_cluster[-1]]) 
                drop = feat_cluster[:-1] 
            else:
                test_feat = other_features
                drop = i
            
            y_true, y_pred, _ = self.model.rolling_forecast(X[test_feat], y)
            err = pinball_loss(y_true, y_pred, 0.5)
            
            if err < best_err:
                best_err = err
                keep_features[drop] = False
                best_features = self.features[sorted_features[keep_features]]
                if verbose:
                    print(f"New best error {best_err:.4f} with features: {best_features}")
        
        return best_features, best_err
                